# 🔍 Regular Expressions from Scratch

A regular expression (**regex**) is a tiny language for describing *patterns in text*: "three digits, a dash, four digits", "a valid IP address", "an email-ish thing". This notebook starts with matching a single digit and ends with parsing long, mixed-format strings and using regex inside pandas.

### How this notebook works

Every topic follows the same loop, so you get lots of repetition:

1. **📖 Learn** – a short explanation.
2. **👀 See it** – a worked example you just run and read.
3. **✏️ Your turn** – several exercises. Each one has a `pattern = r""` stub for you to fill in, and a **checker** underneath that prints ✅ or ❌ with the cases you got wrong.

Each part reuses what the previous one taught, so by Part 11 you're assembling patterns for long strings from small pieces you already wrote.

> **Rules of the road**
> * Run the **setup cell** below first, then go top to bottom.
> * Stuck? Read the `# Hint:` comment, then re-read the "See it" cells above. The **answer key** is at the very bottom. Try before peeking.
> * Exercises marked ⭐ are bonus. Do them if you want extra reps.
> * Go ahead and break things. Change the demo patterns and see what happens.

### Roadmap

| Part | Topic |
|---|---|
| 1 | Literal matching & the `re` functions |
| 2 | Digits: `\d` |
| 3 | Repetition: `+ * ? {n,m}` |
| 4 | Letters, whitespace & character classes |
| 5 | Mixing it all: real formats (postal codes, phones, IDs) |
| 6 | Anchors, word boundaries & flags |
| 7 | Groups, alternation & capturing |
| 8 | Replacing, splitting & greedy vs lazy |
| 9 | Lookarounds (e.g. password rules) |
| 10 | Hex, MAC, IPv4/IPv6, email, URL |
| 11 | Long mixed-format strings (logs, records, tokenizers) |
| 12 | Regex + pandas |

## Setup (run this first)

This cell imports `re` and defines the helper functions used by the checkers. You don't need to read the helpers to use them. They just run your pattern against test cases and report back.

In [ ]:
import re

# ---------- demo helper ----------
def try_it(pattern, strings, flags=0):
    """Demo helper: show which strings match `pattern` IN FULL."""
    for s in strings:
        ok = re.fullmatch(pattern, s, flags) is not None
        print("✔ match   " if ok else "✘ no match", repr(s))

# ---------- checkers ----------
def _report(problems, total):
    if not problems:
        print(f"✅ Nice! All {total} checks passed.")
    else:
        print(f"❌ {len(problems)} of {total} checks failed:")
        for p in problems:
            print(p)

def _compile(pattern, flags=0):
    try:
        return re.compile(pattern, flags)
    except re.error as e:
        print(f"❌ That isn't a valid regex yet: {e}")
        return None

def check_fullmatch(pattern, should_match, should_not_match, flags=0, label=None):
    """Validation: the pattern must match the ENTIRE string (re.fullmatch)."""
    if label:
        print(f"── {label} ──")
    rx = _compile(pattern, flags)
    if rx is None:
        return
    problems = []
    for s in should_match:
        if not rx.fullmatch(s):
            problems.append(f"   should MATCH but didn't     -> {s!r}")
    for s in should_not_match:
        if rx.fullmatch(s):
            problems.append(f"   should be REJECTED but matched -> {s!r}")
    _report(problems, len(should_match) + len(should_not_match))

def check_search(pattern, should_find, should_not_find, flags=0):
    """The pattern must be FOUND somewhere in each 'good' string (re.search)."""
    rx = _compile(pattern, flags)
    if rx is None:
        return
    problems = []
    for s in should_find:
        if not rx.search(s):
            problems.append(f"   should be FOUND but wasn't   -> {s!r}")
    for s in should_not_find:
        if rx.search(s):
            problems.append(f"   should NOT be found but was  -> {s!r}")
    _report(problems, len(should_find) + len(should_not_find))

def _compare(got, expected, what):
    if got == expected:
        _report([], 1)
    else:
        _report([f"   expected {expected!r}\n   got      {got!r}"], 1)

def check_findall(pattern, text, expected, flags=0):
    rx = _compile(pattern, flags)
    if rx is not None:
        _compare(rx.findall(text), expected, "findall")

def check_sub(pattern, repl, text, expected, flags=0):
    rx = _compile(pattern, flags)
    if rx is not None:
        _compare(rx.sub(repl, text), expected, "sub")

def check_split(pattern, text, expected, flags=0):
    rx = _compile(pattern, flags)
    if rx is not None:
        _compare(rx.split(text), expected, "split")

def check_groups(pattern, text, expected, flags=0):
    """Compare m.groups() of re.search(pattern, text) with a tuple."""
    rx = _compile(pattern, flags)
    if rx is None:
        return
    m = rx.search(text)
    _compare(m.groups() if m else None, expected, "groups")

def check_named(pattern, text, expected, flags=0):
    """Compare m.groupdict() of re.search(pattern, text) with a dict."""
    rx = _compile(pattern, flags)
    if rx is None:
        return
    m = rx.search(text)
    _compare(m.groupdict() if m else None, expected, "groupdict")

def check_tokens(pattern, text, expected, flags=0):
    """Compare [(group_name, matched_text), ...] from re.finditer."""
    rx = _compile(pattern, flags)
    if rx is None:
        return
    got = [(m.lastgroup, m.group()) for m in rx.finditer(text)]
    _compare(got, expected, "tokens")

print("Setup complete ✔")

---
# Part 1 · Literal matching & the `re` functions

## 📖 Learn

The simplest regex is just the text you're looking for: `cat` matches the characters `c`, `a`, `t` in that order.

**Always write patterns as raw strings: `r"..."`.** In a normal Python string, `"\n"` is a newline and `"\d"` is a warning. In a raw string, the backslash is left alone for the regex engine to use.

The `re` module has a handful of functions. Learn these seven, because you'll use them for the rest of the notebook:

| Function | Question it answers | Returns |
|---|---|---|
| `re.search(p, s)` | Is the pattern *anywhere* in `s`? | first `Match`, or `None` |
| `re.match(p, s)` | Does `s` *start with* the pattern? | `Match` or `None` |
| `re.fullmatch(p, s)` | Is the *entire* string the pattern? (**validation**) | `Match` or `None` |
| `re.findall(p, s)` | Give me *every* match | list of strings |
| `re.finditer(p, s)` | Same, but as `Match` objects | iterator |
| `re.sub(p, repl, s)` | Replace matches | new string |
| `re.split(p, s)` | Cut `s` at every match | list of strings |

## 👀 See it

In [ ]:
text = "The cat sat on the mat."

m = re.search(r"cat", text)
print(m)                    # a Match object (or None if nothing was found)
print(m.group())            # the text that matched
print(m.start(), m.end())   # where in the string it was found

In [ ]:
s = "cat and cat and cat"

print("search   :", re.search(r"cat", s))      # found at the start
print("match    :", re.match(r"cat", s))       # does it START with cat? yes
print("match    :", re.match(r"and", s))       # does it START with 'and'? no -> None
print("fullmatch:", re.fullmatch(r"cat", s))   # is the WHOLE string 'cat'? no -> None
print("findall  :", re.findall(r"cat", s))
print("sub      :", re.sub(r"cat", "dog", s))
print("split    :", re.split(r" and ", s))

### Special characters

A few characters are **metacharacters** with special meaning: `. ^ $ * + ? { } [ ] \ | ( )`. The most important one right now is `.`, which matches **any single character**. To match a literal dot, put a backslash in front: `\.`

In [ ]:
print(re.findall(r"a.c", "abc a.c a-c ac"))    # . means ANY one character
print(re.findall(r"a\.c", "abc a.c a-c ac"))   # \. means a literal dot

# Not sure which characters to escape? Let Python do it:
print(re.escape("price (USD) = $5.00?"))

## ✏️ Your turn

Fill in each `pattern`. The checker underneath tells you if it's right.

In [ ]:
# ✏️ Exercise 1.1 — Find every "at"
pattern = r""   # <- your pattern here
check_findall(pattern, "The cat sat on the mat with a bat.", ["at", "at", "at", "at"])

In [ ]:
# ✏️ Exercise 1.2 — Match the literal number 3.14
# Hint: a bare `.` matches ANY character, so it needs a backslash.
pattern = r""   # <- your pattern here
check_findall(pattern, "pi is 3.14 but 3x14 is not pi", ["3.14"])

In [ ]:
# ✏️ Exercise 1.3 — Match the literal text (555) including the parentheses
# Hint: ( and ) are metacharacters too.
pattern = r""   # <- your pattern here
check_findall(pattern, "Call (555) 123-4567 or 555 999-0000", ["(555)"])

In [ ]:
# ✏️ Exercise 1.4 — Use the wildcard: match "c", then ANY one character, then "t"
# Should find cat, cot and cut but NOT cart or ct.
pattern = r""   # <- your pattern here
check_findall(pattern, "cat cot cut cart ct", ["cat", "cot", "cut"])

---
# Part 2 · Digits: `\d`

## 📖 Learn

Literal characters are rigid. **Character shorthands** let one pattern match a *kind* of character.

| Shorthand | Matches |
|---|---|
| `\d` | one digit `0-9` |
| `\D` | one character that is **not** a digit |

`\d` matches exactly **one** digit. To match "two digits in a row" you write `\d\d`.

## 👀 See it

In [ ]:
text = "Order 66 shipped on 2024-11-12"

print(re.findall(r"\d", text))      # every single digit, one at a time
print(re.findall(r"\d\d", text))    # two digits in a row (matches don't overlap)
print(re.findall(r"\D", "a1b2c3"))  # \D is the opposite of \d

In [ ]:
# \d can be mixed with literals. "digit digit digit - digit digit digit digit"
print(re.findall(r"\d\d\d-\d\d\d\d", "Call 555-1234 now, not 55-123."))

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 2.1 — Find every single digit
pattern = r""   # <- your pattern here
check_findall(pattern, "Room 4B, floor 2, seat 17", ["4", "2", "1", "7"])

In [ ]:
# ✏️ Exercise 2.2 — Find every pair of digits in a row
# Remember: matches don't overlap, so "1987" gives "19" and "87".
pattern = r""   # <- your pattern here
check_findall(pattern, "Born 1987, aged 36 in 2023, ID 5", ["19", "87", "36", "20", "23"])

In [ ]:
# ✏️ Exercise 2.3 — Find phone numbers shaped like 555-1234
# (three digits, a dash, four digits)
pattern = r""   # <- your pattern here
check_findall(pattern, "Call 555-1234 or 555-9876 today, not 55-123.", ["555-1234", "555-9876"])

In [ ]:
# ✏️ Exercise 2.4 — Find every character that is NOT a digit
pattern = r""   # <- your pattern here
check_findall(pattern, "a1b22c", ["a", "b", "c"])

---
# Part 3 · Repetition: `+ * ? {n,m}`

## 📖 Learn

Writing `\d\d\d\d\d\d` gets old fast. **Quantifiers** say how many times the thing *just before them* may repeat:

| Quantifier | Meaning |
|---|---|
| `+` | one or more |
| `*` | zero or more |
| `?` | zero or one (i.e. *optional*) |
| `{3}` | exactly 3 |
| `{2,4}` | between 2 and 4 |
| `{2,}` | 2 or more |

So `\d+` means "a whole number of any length", and `-?` means "an optional minus/dash".

**Validation vs. searching.** `findall` hunts for matches *inside* a longer string. When you want to know "is this *entire* string a valid ZIP code?", use `re.fullmatch`. From here on, validation exercises use `fullmatch`, and the checker will say which strings should pass and which should fail.

## 👀 See it

In [ ]:
text = "I have 12 apples, 345 oranges and 7 pears."
print(re.findall(r"\d+", text))        # whole numbers of any length
print(re.findall(r"\d{3}", text))      # exactly three digits in a row

In [ ]:
# fullmatch + quantifiers = validation. `-?` makes the dash optional.
try_it(r"\d{3}-?\d{4}", ["555-1234", "5551234", "555--1234", "55-12345"])

In [ ]:
# * allows zero repeats, + needs at least one
print(re.findall(r"Item\d*", "Item Item1 Item22 Items"))   # \d* -> zero or more digits
print(re.findall(r"Item\d+", "Item Item1 Item22 Items"))   # \d+ -> at least one digit

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 3.1 — Find every whole number (any number of digits)
pattern = r""   # <- your pattern here
check_findall(pattern, "Scores: 7, 42, 1003, and 9", ["7", "42", "1003", "9"])

In [ ]:
# ✏️ Exercise 3.2 — Find the four-digit years (exactly 4 digits)
# (Cheat: here the text has no 5-digit numbers, so \d{4} is enough.)
pattern = r""   # <- your pattern here
check_findall(pattern, "Years: 1999, 2024, 87, 305", ["1999", "2024"])

In [ ]:
# ✏️ Exercise 3.3 — Validate a phone number with an OPTIONAL dash: 555-1234 or 5551234
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["555-1234", "5551234"],
    should_not_match=["555--1234", "55-12345", "555 1234", "5551-234"],
)

In [ ]:
# ✏️ Exercise 3.4 — Validate a number with 1 to 3 digits
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["1", "42", "999"],
    should_not_match=["", "1000", "12a", "-5"],
)

In [ ]:
# ✏️ Exercise 3.5 — `?` works on letters too. Match both spellings: color / colour
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["color", "colour"],
    should_not_match=["colouur", "colr", "colors"],
)

---
# Part 4 · Letters, whitespace & character classes

## 📖 Learn

Digits were one *kind* of character. Here are more kinds:

| Pattern | Matches |
|---|---|
| `\w` | a "word" character: letter, digit or underscore |
| `\W` | anything that is **not** a word character |
| `\s` | whitespace (space, tab, newline) |
| `\S` | anything that is **not** whitespace |
| `[abc]` | **one** of `a`, `b`, `c` (a *character class*) |
| `[a-z]` | one lowercase letter (a range) |
| `[A-Za-z0-9]` | ranges can be combined |
| `[^abc]` | one character that is **not** `a`, `b` or `c` |

Inside `[...]`, most metacharacters lose their special meaning, so `[.]` is a literal dot. Quantifiers work on classes too: `[a-z]+` is "one or more lowercase letters".

## 👀 See it

In [ ]:
text = "Hello World from Python 3"

print(re.findall(r"[A-Z]", text))       # capital letters
print(re.findall(r"[a-z]+", text))      # runs of lowercase letters
print(re.findall(r"\w+", text))         # "words": letters/digits/underscore
print(re.findall(r"[^ ]+", text))       # runs of anything-but-a-space
print(re.findall(r"[aeiou]", text))     # a hand-made class: vowels

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 4.1 — Find every capital letter
pattern = r""   # <- your pattern here
check_findall(pattern, "Hello World From Python", ["H", "W", "F", "P"])

In [ ]:
# ✏️ Exercise 4.2 — Find every run of lowercase letters
pattern = r""   # <- your pattern here
check_findall(pattern, "hello WORLD from python", ["hello", "from", "python"])

In [ ]:
# ✏️ Exercise 4.3 — Find every vowel (a, e, i, o, u)
pattern = r""   # <- your pattern here
check_findall(pattern, "regular expressions", ["e", "u", "a", "e", "e", "i", "o"])

In [ ]:
# ✏️ Exercise 4.4 — Find every "word" (\w covers letters, digits and underscores)
pattern = r""   # <- your pattern here
check_findall(pattern, "snake_case and camelCase, 2 words!", ["snake_case", "and", "camelCase", "2", "words"])

In [ ]:
# ✏️ Exercise 4.5 — Split a comma-separated line by matching the FIELDS:
# one or more characters that are NOT a comma.
pattern = r""   # <- your pattern here
check_findall(pattern, "a,b,,c", ["a", "b", "c"])

---
# Part 5 · Mixing it all: real formats

## 📖 Learn

Real-world formats are just **literals + character classes + quantifiers** lined up in order. The trick is to read the format left to right and write one "slot" at a time:

```
ABC-1234      →   [A-Z]{3}   -   \d{4}
              3 capitals   dash   4 digits
```

Whenever you build a pattern, ask for each slot: *what kind of character? how many?*

## 👀 See it

In [ ]:
# Product code: 3 capital letters, a dash, 4 digits
try_it(r"[A-Z]{3}-\d{4}", ["ABC-1234", "XYZ-0001", "abc-1234", "AB-1234", "ABC-123", "ABCD-1234"])

In [ ]:
# Same pattern, now used to *extract* codes from a sentence
text = "Shipped ABC-1234 and XYZ-0001; ignore abc-9999 and AB-12."
print(re.findall(r"[A-Z]{3}-\d{4}", text))

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 5.1 — Canadian postal code: letter digit letter, a space, digit letter digit
# e.g. K1A 0B1  (capital letters only)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["K1A 0B1", "M5V 2T6"],
    should_not_match=["k1a 0b1", "K1A0B1", "K11 0B1", "K1A 0B"],
)

In [ ]:
# ✏️ Exercise 5.2 — US phone number in the form (555) 123-4567
# Hint: escape the parentheses.
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["(555) 123-4567", "(800) 000-0000"],
    should_not_match=["555-123-4567", "(555)123-4567", "(55) 123-4567", "(555) 123-456"],
)

In [ ]:
# ✏️ Exercise 5.3 — ISO date: YYYY-MM-DD (just the shape, not whether the month is real)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["2024-11-12", "1999-01-31"],
    should_not_match=["24-11-12", "2024/11/12", "2024-1-12", "2024-11-123"],
)

In [ ]:
# ✏️ Exercise 5.4 — Username: starts with a letter, then letters/digits/underscores,
# total length 3 to 12.
# Hint: the first character is one slot, the rest is another slot with a {min,max}.
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["bob", "alice_99", "Z9_", "user_name_12"],
    should_not_match=["ab", "9lives", "has space", "waytoolongusername1", "no-dash"],
)

In [ ]:
# ✏️ Exercise 5.5 — Extract order IDs like ORD-20241112-0042 from a sentence
# (ORD, dash, 8 digits, dash, 4 digits)
pattern = r""   # <- your pattern here
text = "Orders ORD-20241112-0042 and ORD-20241113-0001 shipped; ORD-2024-1 is pending."
check_findall(pattern, text, ["ORD-20241112-0042", "ORD-20241113-0001"])

---
# Part 6 · Anchors, word boundaries & flags

## 📖 Learn

So far patterns matched *wherever* they fit. **Anchors** pin a pattern to a position. They match a *position*, not a character:

| Anchor | Meaning |
|---|---|
| `^` | start of the string |
| `$` | end of the string |
| `\b` | a **word boundary** (between a `\w` and a non-`\w`) |

`re.fullmatch(p, s)` behaves like `re.search("^" + p + "$", s)`. Anchors let you do the same inside `search`/`findall`, and also pin just *one* end ("starts with", "ends with").

**Flags** change how a whole pattern behaves. The one you'll use most is `re.IGNORECASE` (or `re.I`). Pass it as the last argument: `re.findall(p, s, re.I)`.

## 👀 See it

In [ ]:
print(re.findall(r"cat", "cat concatenate cats"))          # matches inside other words too!
print(re.findall(r"\bcat\b", "cat concatenate cats"))     # only the standalone word

print(bool(re.search(r"^ERROR", "ERROR: disk full")))      # starts with ERROR?
print(bool(re.search(r"^ERROR", "Got an ERROR")))          # ...not here
print(bool(re.search(r"\.csv$", "data.csv")))              # ends with .csv?

print(re.findall(r"python", "Python PYTHON python", re.IGNORECASE))

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 6.1 — Find the standalone word "cat" (not "concat" or "cats")
pattern = r""   # <- your pattern here
check_findall(pattern, "cat concat cats cat.", ["cat", "cat"])

In [ ]:
# ✏️ Exercise 6.2 — Find numbers that are exactly 3 digits long (whole numbers only,
# so don't pull 123 out of 1234). Hint: \b on both sides.
pattern = r""   # <- your pattern here
check_findall(pattern, "12 123 1234 987 45", ["123", "987"])

In [ ]:
# ✏️ Exercise 6.3 — Find the word "python" in any capitalisation, as a standalone word.
# Fill in BOTH the pattern and the flags.
pattern = r""   # <- your pattern here
flags = 0     # <- which flag makes matching case-insensitive?
check_findall(pattern, "Python, PYTHON, pythonic, python3 and python.", ["Python", "PYTHON", "python"], flags)

In [ ]:
# ✏️ Exercise 6.4 — Detect log lines that START with ERROR
pattern = r""   # <- your pattern here
check_search(
    pattern,
    should_find=["ERROR: disk full", "ERROR"],
    should_not_find=["WARN: ERROR later", "  ERROR indented", "error: lower"],
)

In [ ]:
# ✏️ Exercise 6.5 — Detect file names that END with .csv
pattern = r""   # <- your pattern here
check_search(
    pattern,
    should_find=["data.csv", "a.b.csv"],
    should_not_find=["data.csv.bak", "csv", "data.CSV"],
)

---
# Part 7 · Groups, alternation & capturing

## 📖 Learn

Parentheses `( )` do **three** jobs:

1. **Group** things so a quantifier applies to the whole group: `(ab)+` matches `ab`, `abab`, ...
2. **Capture** what the group matched so you can pull it out with `m.group(1)` / `m.groups()`.
3. **Name** a capture with `(?P<name>...)` so you can use `m.group("name")` or `m.groupdict()`.

Other tools:

| Syntax | Meaning |
|---|---|
| `a\|b` | alternation: `a` **or** `b` |
| `(?:...)` | a **non-capturing** group (grouping only, no capture) |
| `\1` | a **backreference**: "the same text group 1 matched" |

> ⚠️ `re.findall` behaves differently when the pattern has capture groups: it returns the *groups* (tuples) instead of the whole match. Use `(?:...)` when you only want grouping.

## 👀 See it

In [ ]:
# 1. grouping + quantifier, 2. alternation
try_it(r"(ab)+", ["ab", "abab", "aba", "ba"])
try_it(r"\w+\.(?:jpg|png)", ["cat.jpg", "dog.png", "cat.gif"])

In [ ]:
# 3. capturing
m = re.search(r"(\d{4})-(\d{2})-(\d{2})", "Due on 2024-11-12, thanks")
print(m.groups())       # ('2024', '11', '12')
print(m.group(1))       # just the year

# 4. named groups are self-documenting
m = re.search(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", "Due on 2024-11-12")
print(m.groupdict())

# 5. findall + groups returns tuples!
print(re.findall(r"(\d{4})-(\d{2})", "2024-11 and 2025-01"))
print(re.findall(r"(?:\d{4})-(?:\d{2})", "2024-11 and 2025-01"))   # non-capturing: whole matches

# 6. backreference: the same word twice in a row
print(re.search(r"\b(\w+) \1\b", "this is is a typo").group())

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 7.1 — Alternation: an image file name ending in .jpg, .png or .gif
# (lowercase extensions; name is word characters)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["cat.jpg", "dog.png", "a_b.gif"],
    should_not_match=["cat.jpeg", "cat.", "cat.JPG", "cat.jpgx"],
)

In [ ]:
# ✏️ Exercise 7.2 — Grouping: laughter! One or more "ha" followed by a "!"
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["ha!", "haha!", "hahaha!"],
    should_not_match=["!", "hah!", "ha", "haHa!"],
)

In [ ]:
# ✏️ Exercise 7.3 — Capture the hour and minute from "Meeting at 14:30"
# Use two capture groups so that m.groups() == ("14", "30")
pattern = r""   # <- your pattern here
check_groups(pattern, "Meeting at 14:30", ("14", "30"))
check_groups(pattern, "Lunch at 9:05 sharp", ("9", "05"))

In [ ]:
# ✏️ Exercise 7.4 — Named groups: parse "user=alice id=42"
# Name the groups  user  and  id.
pattern = r""   # <- your pattern here
check_named(pattern, "user=alice id=42", {"user": "alice", "id": "42"})
check_named(pattern, "user=bob_7 id=1001", {"user": "bob_7", "id": "1001"})

In [ ]:
# ✏️ Exercise 7.5 — Optional group: a US phone with an OPTIONAL "+1-" prefix
# e.g. 555-123-4567 or +1-555-123-4567
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["555-123-4567", "+1-555-123-4567"],
    should_not_match=["+1555-123-4567", "555-123-456", "+2-555-123-4567"],
)

In [ ]:
# ✏️ Exercise 7.6 ⭐ — Backreference: detect an accidentally repeated word ("the the")
# Use a capture group for the word and \1 to repeat it.
pattern = r""   # <- your pattern here
check_search(
    pattern,
    should_find=["this is is a test", "the the"],
    should_not_find=["this is a test", "the them"],
)

---
# Part 8 · Replacing, splitting & greedy vs lazy

## 📖 Learn

* `re.sub(pattern, replacement, text)` replaces every match. In the replacement, `\1`, `\2`, ... insert captured groups, so you can **reorder** text.
* `re.split(pattern, text)` cuts the string wherever the pattern matches. This is `str.split` for messy separators.
* **Greedy vs lazy:** quantifiers are *greedy*, so they grab as much as possible. Add `?` after one (`+?`, `*?`) to make it *lazy*, so it grabs as little as possible.

## 👀 See it

In [ ]:
print(re.sub(r"\d+", "#", "Card 1234 5678"))                          # simple replace
print(re.sub(r"(\w+), (\w+)", r"\2 \1", "Smith, John"))               # swap using groups
print(re.split(r"\s*[,;]\s*", "a , b;c ,d"))                          # messy separators

html = "<b>bold</b> and <i>italic</i>"
print(re.findall(r"<.+>", html))     # greedy: swallows everything between first < and last >
print(re.findall(r"<.+?>", html))    # lazy: stops at the first >

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 8.1 — Redact: replace every run of digits with a single #
pattern = r""   # <- your pattern here
repl = ""      # <- your replacement here
check_sub(pattern, repl, "Card 1234 5678 expires 12/26", "Card # # expires #/#")

In [ ]:
# ✏️ Exercise 8.2 — Reformat dates from YYYY-MM-DD to DD/MM/YYYY
# Capture year, month, day; then rebuild with \3 \2 \1 in the replacement.
pattern = r""   # <- your pattern here
repl = ""      # <- your replacement here (a raw string like r"\1-\2")
check_sub(pattern, repl, "Start 2024-11-12, end 2025-01-05", "Start 12/11/2024, end 05/01/2025")

In [ ]:
# ✏️ Exercise 8.3 — Split on a comma, semicolon or pipe, ignoring spaces around it
pattern = r""   # <- your pattern here
check_split(pattern, "a , b;c|  d", ["a", "b", "c", "d"])

In [ ]:
# ✏️ Exercise 8.4 — Lazy matching: pull out each HTML tag separately
# Hint: add ? after the quantifier, or use [^>]+ instead of .+
pattern = r""   # <- your pattern here
check_findall(pattern, "<b>bold</b> and <i>it</i>", ["<b>", "</b>", "<i>", "</i>"])

In [ ]:
# ✏️ Exercise 8.5 — Normalise whitespace: collapse any run of spaces/tabs/newlines into ONE space
pattern = r""   # <- your pattern here
repl = ""      # <- your replacement here
check_sub(pattern, repl, "too    many\t\tspaces\nhere", "too many spaces here")

---
# Part 9 · Lookarounds

## 📖 Learn

A **lookaround** checks what's next to the current position *without consuming it*. It matches a position, like `^` or `\b`, so what it looks at is not part of the match.

| Syntax | Name | Meaning |
|---|---|---|
| `(?=...)` | positive lookahead | what follows must match `...` |
| `(?!...)` | negative lookahead | what follows must **not** match `...` |
| `(?<=...)` | positive lookbehind | what precedes must match `...` |
| `(?<!...)` | negative lookbehind | what precedes must **not** match `...` |

The classic use is **stacking requirements** at the start of a pattern: each `(?=...)` is an independent check on the same string.

## 👀 See it

In [ ]:
# numbers that are followed by " kg" (but don't include the kg)
print(re.findall(r"\d+(?= kg)", "5 kg, 12 lbs, 70 kg"))

# digits preceded by a dollar sign
print(re.findall(r"(?<=\$)\d+", "cost $15 or 20 euros"))

# stacked lookaheads: "contains a digit" AND "contains a lowercase letter", length >= 6
rule = r"(?=.*\d)(?=.*[a-z]).{6,}"
try_it(rule, ["abc123", "abcdef", "123456", "ab1"])

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 9.1 — Find numbers followed by " dollars" (don't include the word)
pattern = r""   # <- your pattern here
check_findall(pattern, "Pay 50 dollars, 30 euros, 20 dollars", ["50", "20"])

In [ ]:
# ✏️ Exercise 9.2 — Find prices after a $ sign: whole dollars with optional .cents
# Return only the number (no $). Careful: findall returns GROUPS if you capture,
# so use (?:...) for the optional cents part.
pattern = r""   # <- your pattern here
check_findall(pattern, "Cost $15.99 and $8 plus 20 cents", ["15.99", "8"])

In [ ]:
# ✏️ Exercise 9.3 — Password rule (validate with fullmatch):
#   at least 8 characters, at least one digit, one lowercase and one uppercase letter.
# Hint: three stacked lookaheads (?=.*...) and then .{8,}
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["Passw0rdOK", "Abcdefg1", "LongPassword123"],
    should_not_match=["short1A", "alllowercase1", "ALLUPPERCASE1", "NoDigitsHere"],
)

In [ ]:
# ✏️ Exercise 9.4 — Negative lookahead: find words that do NOT start with "un"
pattern = r""   # <- your pattern here
check_findall(pattern, "happy unhappy usual unusual", ["happy", "usual"])

---
# Part 10 · Hex, MAC, IPv4/IPv6, email & URL

## 📖 Learn

Time to use everything on formats you'll actually meet. Two techniques will carry you:

**1. Hex digits** are `[0-9a-fA-F]`. A hex byte is `[0-9a-fA-F]{2}`.

**2. Build numeric ranges one leading digit at a time.** There's no "0 to 255" in regex. You write it as alternatives:

| Range | Pattern |
|---|---|
| 0–9 | `\d` |
| 10–99 | `[1-9]\d` |
| 100–199 | `1\d\d` |
| 200–249 | `2[0-4]\d` |
| 250–255 | `25[0-5]` |

Join them with `|`, wrap in `(?:...)`, and you have a number-range matcher. Order doesn't matter when you use `fullmatch`.

## 👀 See it

In [ ]:
# Hex colours and bytes
try_it(r"#[0-9a-fA-F]{6}", ["#FF00aa", "#123456", "#12345", "#GG0000", "123456"])
print()

# Number ranges, built one leading digit at a time: 0-199
below_200 = r"1\d\d|[1-9]?\d"       # 100-199  or  (optional tens digit)(units digit)
try_it(below_200, ["0", "9", "10", "99", "100", "199", "200", "01"])

In [ ]:
# Reusing a piece: build a bigger pattern out of smaller ones with Python strings.
num = r"(?:1\d\d|[1-9]?\d)"                  # 0-199 (wrapped in a group so it can be repeated)
pair = num + r"-" + num                       # "num-num"
try_it(pair, ["5-150", "0-0", "200-1", "5-"])

### 10A · Hex & MAC addresses

In [ ]:
# ✏️ Exercise 10.1 — A hex byte: exactly two hex digits (upper or lower case)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["00", "ff", "A9", "7e"],
    should_not_match=["g1", "123", "f", "0x", ""],
)

In [ ]:
# ✏️ Exercise 10.2 — CSS hex colour: a # followed by EITHER 3 OR 6 hex digits
# Hint: group three hex digits and repeat the group {1,2} times.
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["#fff", "#FF00aa", "#123456", "#abc"],
    should_not_match=["fff", "#ffff", "#12345", "#ggg", "#1234567"],
)

In [ ]:
# ✏️ Exercise 10.3 — A hex memory address: 0x (or 0X) then 1 to 16 hex digits
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["0x1A2B", "0X7fffffff", "0xdeadbeef", "0x0", "0x7FFE1A20"],
    should_not_match=["1A2B", "0x", "0xGHI", "0x12345678901234567"],
)

In [ ]:
# ✏️ Exercise 10.4 — A MAC address: six hex bytes separated by colons OR dashes
# e.g. 00:1A:2B:3C:4D:5E  or  aa-bb-cc-dd-ee-ff
# Hint: "byte + separator" repeated 5 times, then one last byte.
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["00:1A:2B:3C:4D:5E", "aa-bb-cc-dd-ee-ff"],
    should_not_match=["00:1A:2B:3C:4D", "00:1A:2B:3C:4D:5E:6F", "001A.2B3C.4D5E", "00:1A:2B:3C:4D:5G"],
)

In [ ]:
# ✏️ Exercise 10.5 ⭐ — Same MAC address, but the separator must be CONSISTENT.
# 00:1A:2B:3C:4D:5E is fine, 00:1A-2B:3C:4D:5E is not.
# Hint: capture the first separator ([:-]) and reuse it with \1.
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["00:1A:2B:3C:4D:5E", "aa-bb-cc-dd-ee-ff"],
    should_not_match=["00:1A-2B:3C:4D:5E", "00:1A:2B:3C:4D", "00-1A-2B-3C-4D-5E-6F"],
)

### 10B · IP addresses

In [ ]:
# ✏️ Exercise 10.6 — A LOOSE IPv4 address: four groups of 1-3 digits separated by dots
# (Doesn't check the 0-255 range yet. That's next.)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["192.168.1.1", "10.0.0.255", "8.8.8.8", "999.999.999.999"],
    should_not_match=["192.168.1", "1.2.3.4.5", "a.b.c.d", "192.168..1"],
)

In [ ]:
# ✏️ Exercise 10.7 — One IPv4 octet: a number from 0 to 255, no leading zeros
# The demo above gave you 0-199 as  1\d\d|[1-9]?\d  . Extend it to 255 by adding
# the 200-249 and 250-255 alternatives.
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["0", "9", "10", "99", "100", "199", "200", "249", "250", "255"],
    should_not_match=["256", "300", "01", "1000", "-1", "", " 5"],
)

In [ ]:
# ✏️ Exercise 10.8 — A STRICT IPv4 address. We give you `octet` (your 10.7 answer, grouped).
# Combine it: octet, then (dot + octet) three times. Use string concatenation.
octet = r"(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)"
pattern = r""   # <- your pattern here, e.g. octet + r"..." + ...
check_fullmatch(
    pattern,
    should_match=["192.168.1.1", "0.0.0.0", "255.255.255.255", "10.0.0.1", "8.8.8.8"],
    should_not_match=["256.1.1.1", "192.168.1", "1.2.3.4.5", "01.1.1.1", "192.168.1.300", "1.1.1.1 ", "a.b.c.d"],
)

In [ ]:
# ✏️ Exercise 10.9 — IPv4 with an OPTIONAL CIDR suffix: /0 to /32  (e.g. 192.168.1.0/24)
# Prefix lengths 0-32:  3[0-2] | [12]?\d
octet = r"(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)"
ipv4  = octet + r"(?:\." + octet + r"){3}"
pattern = r""   # <- your pattern here: ipv4 + (optional group with / and the prefix)
check_fullmatch(
    pattern,
    should_match=["192.168.1.0/24", "10.0.0.0/8", "8.8.8.8", "172.16.0.0/12", "1.1.1.1/32", "1.1.1.1/0"],
    should_not_match=["192.168.1.0/33", "192.168.1.0/", "10.0.0.0/-1", "1.1.1.1/100", "256.1.1.1/8"],
)

In [ ]:
# ✏️ Exercise 10.10 — Full-form IPv6: eight groups of 1-4 hex digits separated by colons
# (we ignore the :: shorthand)
# e.g. 2001:0db8:85a3:0000:0000:8a2e:0370:7334
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["2001:0db8:85a3:0000:0000:8a2e:0370:7334", "fe80:0:0:0:1:2:3:4"],
    should_not_match=[
        "2001:0db8:85a3:0000:0000:8a2e:0370",            # only 7 groups
        "2001:0db8:85a3:0000:0000:8a2e:0370:7334:1234",  # 9 groups
        "2001:0db8:85a3:0000:0000:8a2e:0370:gggg",       # not hex
        "12345:0:0:0:0:0:0:1",                           # 5 digits in a group
    ],
)

### 10C · Email & URL (simplified)

Real-world email/URL validation is famously messy. These are *good-enough* versions built from the same slots you already know.

In [ ]:
# ✏️ Exercise 10.11 — A simplified email address:
#   local part: letters, digits, _ . + -   (one or more)
#   then @
#   domain: labels of letters/digits/hyphens, separated by dots, at least one dot
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["alice@example.com", "bob.smith+tag@mail.co.uk", "x_y@sub.domain.org"],
    should_not_match=["alice@", "@example.com", "alice@example", "alice example@x.com", "alice@@x.com", "alice@.com"],
)

In [ ]:
# ✏️ Exercise 10.12 — A simplified URL, built from optional parts:
#   scheme    http:// or https://
#   host      letters/digits/dots/hyphens (one or more)
#   port      optional  :1-5 digits
#   path      optional  /  followed by letters, digits, . / % -   (can be just "/")
#   query     optional  ?  followed by letters, digits, = & % .
# Hint: build the pieces as separate Python strings and join them.
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=[
        "http://example.com",
        "http://example.com/",
        "https://example.com:8080/path/to/page",
        "https://sub.example.co.uk/a/b.html?x=1&y=2",
        "http://localhost:3000",
    ],
    should_not_match=["ftp://example.com", "http//example.com", "https://", "http://exa mple.com", "example.com"],
)

---
# Part 11 · Long mixed-format strings

## 📖 Learn

Real data is a *sequence* of formats glued together: a timestamp, then an IP, then a status code... The way to stay sane:

1. **Build each field as its own small pattern** and test it alone.
2. **Name the fields** with `(?P<name>...)` so the result is readable.
3. **Use `re.VERBOSE`** (`re.X`) so you can lay a long pattern out over several lines with comments. In verbose mode, whitespace in the pattern is ignored (use `\s` or `\ ` for a real space), and `#` starts a comment (use `\#` for a literal one).
4. **Assemble** the pieces with Python (`"|".join(...)`, `+`, f-strings).

## 👀 See it

In [ ]:
alert_pattern = r"""
    ^ALERT \s+
    sensor=(?P<sensor>\d+) \s+                          # sensor number
    temp=(?P<temp>-?\d+(?:\.\d+)?)(?P<unit>[CF]) \s+    # e.g. 71.5F
    at \s+ (?P<hour>\d{2}):(?P<minute>\d{2})            # 24h time
    $
"""

m = re.search(alert_pattern, "ALERT sensor=7 temp=71.5F at 14:02", re.VERBOSE)
print(m.groupdict())

In [ ]:
# A "tokenizer": one big alternation of NAMED groups. m.lastgroup tells you which alternative matched.
token_pattern = r"(?P<NUMBER>\d+(?:\.\d+)?)|(?P<WORD>[A-Za-z_]\w*)|(?P<OP>[+\-*/=])"

for m in re.finditer(token_pattern, "total = price*2.5 + tax"):
    print(f"{m.lastgroup:7} {m.group()!r}")

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 11.1 — A STRICT timestamp:  YYYY-MM-DD HH:MM:SS
#   month 01-12 · day 01-31 · hour 00-23 · minute 00-59 · second 00-59
# Hint: build each range like the octet: month = 0[1-9]|1[0-2]  ...
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=["2024-11-12 14:03:22", "1999-01-01 00:00:00", "2024-12-31 23:59:59"],
    should_not_match=[
        "2024-13-12 14:03:22",   # month 13
        "2024-00-10 10:10:10",   # month 00
        "2024-11-32 14:03:22",   # day 32
        "2024-11-00 14:03:22",   # day 00
        "2024-11-12 24:00:00",   # hour 24
        "2024-11-12 14:60:22",   # minute 60
        "2024-11-12 14:03",      # no seconds
    ],
)

In [ ]:
# ✏️ Exercise 11.2 — Parse a web-server log line with NAMED groups and re.VERBOSE.
#   2024-11-12 14:03:22 [ERROR] 192.168.1.15 GET /api/users?id=42 500 123ms
# Replace each ??? with the right sub-pattern. (Until you do, the checker will tell
# you the regex isn't valid yet. That's expected.)
#   date   YYYY-MM-DD        time   HH:MM:SS         level  capital letters
#   ip     loose IPv4        method GET|POST|PUT|DELETE
#   path   non-whitespace    status 3 digits         ms     digits (then the literal "ms")
pattern = r"""
    (?P<date>   ???   )   \s+
    (?P<time>   ???   )   \s+
    \[ (?P<level>   ???   ) \]   \s+
    (?P<ip>   ???   )   \s+
    (?P<method>   ???   )   \s+
    (?P<path>   ???   )   \s+
    (?P<status>   ???   )   \s+
    (?P<ms>   ???   ) ms
"""
line1 = "2024-11-12 14:03:22 [ERROR] 192.168.1.15 GET /api/users?id=42 500 123ms"
line2 = "2024-11-12 14:05:01 [INFO] 10.0.0.7 POST /login 200 87ms"

check_named(pattern, line1, {"date": "2024-11-12", "time": "14:03:22", "level": "ERROR",
                             "ip": "192.168.1.15", "method": "GET", "path": "/api/users?id=42",
                             "status": "500", "ms": "123"}, re.VERBOSE)
check_named(pattern, line2, {"date": "2024-11-12", "time": "14:05:01", "level": "INFO",
                             "ip": "10.0.0.7", "method": "POST", "path": "/login",
                             "status": "200", "ms": "87"}, re.VERBOSE)

In [ ]:
# ✏️ Exercise 11.3 — Extract every VALID IPv4 address from free text (use your strict
# octet logic, plus \b on both ends so you don't pull pieces out of bad addresses).
octet = r"(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)"
pattern = r""   # <- your pattern here
text = "Hosts 192.168.0.1 and 10.0.0.255 were seen; 256.1.1.1 is invalid, 8.8.8.8 is fine."
check_findall(pattern, text, ["192.168.0.1", "10.0.0.255", "8.8.8.8"])

### The big one: a long record with six different formats

A sensor record looks like this (fields separated by `|`):

```
DEV-A1B2C3|2024-11-12T14:03:22Z|192.168.1.15|00:1A:2B:3C:4D:5E|0x7FFE1A20|temp=23.5C
 device id | ISO timestamp (UTC) |   IPv4     |   MAC address    | hex address | reading
```

Don't write it as one giant pattern. Write **one pattern per field** (each is something you've already built!), test each, then let Python glue them together.

In [ ]:
# ✏️ Exercise 11.4 — Build the six field patterns. The checker tests each one, then
# glues them together with "|" separators and tests the whole record.
#
#   dev_id   DEV- followed by exactly 6 UPPERCASE hex digits (0-9, A-F)
#   iso_ts   YYYY-MM-DDTHH:MM:SSZ  (strict month/day/hour/minute/second, literal T and Z)
#   ip       strict IPv4 (we give you `octet`)
#   mac      six hex bytes separated by colons only
#   addr     0x followed by exactly 8 hex digits (upper or lower case)
#   reading  lowercase name, =, optional minus, number with optional decimals, optional unit
#            (letters or %)   e.g. temp=23.5C  humidity=-40%  volts=5
octet   = r"(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)"
dev_id  = r""
iso_ts  = r""
ip      = r""
mac     = r""
addr    = r""
reading = r""

check_fullmatch(dev_id, ["DEV-A1B2C3", "DEV-000000", "DEV-FFFFFF"],
                ["DEV-A1B2C", "DEV-A1B2C3D", "dev-A1B2C3", "DEV-G1B2C3", "A1B2C3"], label="dev_id")
check_fullmatch(iso_ts, ["2024-11-12T14:03:22Z", "1999-01-01T00:00:00Z"],
                ["2024-13-12T14:03:22Z", "2024-11-12 14:03:22Z", "2024-11-12T25:03:22Z", "2024-11-12T14:03:22"], label="iso_ts")
check_fullmatch(ip, ["192.168.1.15", "0.0.0.0", "255.255.255.255"],
                ["300.1.1.1", "1.2.3", "1.2.3.4.5"], label="ip")
check_fullmatch(mac, ["00:1A:2B:3C:4D:5E", "aa:bb:cc:dd:ee:ff"],
                ["00-1A-2B-3C-4D-5E", "00:1A:2B:3C:4D", "GG:1A:2B:3C:4D:5E"], label="mac")
check_fullmatch(addr, ["0x7FFE1A20", "0xdeadbeef", "0x00000000"],
                ["0x7FFE1A2", "0x7FFE1A200", "7FFE1A20", "0xZZZZZZZZ"], label="addr")
check_fullmatch(reading, ["temp=23.5C", "humidity=-40%", "volts=5", "pressure=101.3kPa"],
                ["temp=", "=23.5C", "Temp=23.5C", "temp=23.", "temp 23.5C"], label="reading")

# ---- Glue them together (this part is done for you) ----
record = r"\|".join(f"(?:{p})" for p in [dev_id, iso_ts, ip, mac, addr, reading])

ok = ["DEV-A1B2C3", "2024-11-12T14:03:22Z", "192.168.1.15", "00:1A:2B:3C:4D:5E", "0x7FFE1A20", "temp=23.5C"]
def with_field(i, value):            # a copy of `ok` with field i swapped for a bad value
    parts = ok.copy(); parts[i] = value
    return "|".join(parts)

good_records = [
    "|".join(ok),
    "DEV-00FF9A|1999-01-01T00:00:00Z|10.0.0.1|aa:bb:cc:dd:ee:ff|0xdeadbeef|humidity=-40%",
    "DEV-123ABC|2024-12-31T23:59:59Z|255.255.255.255|01:23:45:67:89:AB|0x00000000|volts=5",
]
bad_records = [
    with_field(0, "DEV-A1B2G3"), with_field(1, "2024-13-12T14:03:22Z"), with_field(2, "300.168.1.15"),
    with_field(3, "00:1A:2B:3C:4D"), with_field(4, "0x7FFE1A2"), with_field(5, "temp="),
    "|".join(ok[:5]),                  # a field is missing
    "|".join(ok) + "|extra=1",         # one field too many
]
check_fullmatch(record, good_records, bad_records, label="WHOLE RECORD")

In [ ]:
# ✏️ Exercise 11.5 ⭐ — A tokenizer for free text. Write ONE pattern that is an alternation
# of NAMED groups, so that re.finditer + m.lastgroup labels every item:
#   ORDER  ORD-########-####     IP    loose IPv4       DATE  YYYY-MM-DD
#   MONEY  $ then digits with optional thousands commas and optional .cents   e.g. $1,249.99
#   HEX    0x followed by hex digits
# Remember: (?P<NAME>...)|(?P<OTHER>...)|...
pattern = r""   # <- your pattern here
text = "Order ORD-20241112-0042 shipped to 192.168.1.15 on 2024-11-12 for $1,249.99 (hex 0xFF)."
check_tokens(pattern, text, [
    ("ORDER", "ORD-20241112-0042"),
    ("IP", "192.168.1.15"),
    ("DATE", "2024-11-12"),
    ("MONEY", "$1,249.99"),
    ("HEX", "0xFF"),
])

---
# Part 12 · Regex + pandas

## 📖 Learn

pandas exposes regex through the `.str` accessor on a Series. The one-line summary:

| Method | What it does | Returns |
|---|---|---|
| `s.str.contains(p)` | pattern found *anywhere*? (like `re.search`) | boolean Series |
| `s.str.match(p)` | pattern matches at the *start*? (like `re.match`) | boolean Series |
| `s.str.fullmatch(p)` | *whole* string matches? (like `re.fullmatch`) | boolean Series |
| `s.str.extract(p)` | pull out the **capture groups** of the first match | DataFrame (one column per group; named groups → column names) |
| `s.str.extractall(p)` | same, for every match | DataFrame with a MultiIndex |
| `s.str.count(p)` | how many matches per string | integer Series |
| `s.str.replace(p, r, regex=True)` | `re.sub` for every row | Series |

Everything you learned in Parts 1–11 works unchanged inside those calls, so this part is mostly *where to put the pattern*. The exercises below only ask you to write the **pattern**. The checker runs the pandas call for you, and the code it runs is shown in a comment.

## 👀 See it

In [ ]:
import pandas as pd

def _show_err(e):
    print(f"❌ pandas raised {type(e).__name__}: {e}")

def check_mask(pattern, series, expected, how="contains"):
    """Runs series.str.<how>(pattern) and compares the True/False list."""
    try:
        got = getattr(series.str, how)(pattern).tolist()
    except Exception as e:
        return _show_err(e)
    _compare(got, expected, how)

def check_replace(pattern, repl, series, expected):
    """Runs series.str.replace(pattern, repl, regex=True)."""
    try:
        got = series.str.replace(pattern, repl, regex=True).tolist()
    except Exception as e:
        return _show_err(e)
    _compare(got, expected, "replace")

def check_extract(pattern, series, expected):
    """Runs series.str.extract(pattern) and compares each named column."""
    try:
        got = series.str.extract(pattern).fillna("").to_dict("list")
    except Exception as e:
        return _show_err(e)
    _compare(got, expected, "extract")

def check_count(pattern, series, expected):
    """Runs series.str.count(pattern)."""
    try:
        got = series.str.count(pattern).tolist()
    except Exception as e:
        return _show_err(e)
    _compare(got, expected, "count")

def check_columns(df, expected):
    """Compare whole columns of a DataFrame (NaN/None are treated as '')."""
    problems = []
    for col, exp in expected.items():
        if col not in df.columns:
            problems.append(f"   missing column {col!r}")
            continue
        got = df[col].fillna("").astype(str).tolist()
        if got != exp:
            problems.append(f"   column {col!r}:\n      expected {exp}\n      got      {got}")
    _report(problems, len(expected))

logs = pd.DataFrame({"raw": [
    "2024-11-12 14:03:22 [ERROR] 192.168.1.15 GET /api/users?id=42 500 123ms",
    "2024-11-12 14:05:01 [INFO] 10.0.0.7 POST /login 200 87ms",
    "2024-11-12 14:07:45 [WARN] 192.168.1.15 GET /api/orders 404 15ms",
    "2024-11-12 14:09:10 [ERROR] 172.16.0.4 PUT /api/items/9 503 2210ms",
    "2024-11-12 14:11:30 [INFO] 10.0.0.7 GET /home 200 45ms",
]})
logs

In [ ]:
# contains / match / fullmatch: boolean masks you can use to filter rows
print(logs["raw"].str.contains(r"\[INFO\]").tolist())          # anywhere in the string
print(logs["raw"].str.match(r"\d{4}-\d{2}-\d{2}").tolist())    # at the START
print(logs["raw"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").tolist())# the WHOLE string (all False)

logs[logs["raw"].str.contains(r"\[ERROR\]")]                   # filter rows with a mask

In [ ]:
# extract: your Part 11 named-group pattern becomes columns. The pattern from 11.2 works as-is.
log_pattern = r"""
    (?P<date>\d{4}-\d{2}-\d{2}) \s+ (?P<time>\d{2}:\d{2}:\d{2}) \s+
    \[(?P<level>[A-Z]+)\] \s+ (?P<ip>\S+) \s+ (?P<method>[A-Z]+) \s+
    (?P<path>\S+) \s+ (?P<status>\d{3}) \s+ (?P<ms>\d+)ms
"""
parsed = logs["raw"].str.extract(log_pattern, flags=re.VERBOSE)
parsed["ms"] = parsed["ms"].astype(int)        # regex gives text; convert the numbers
parsed

In [ ]:
# ...and now it's ordinary pandas: average latency per log level
parsed.groupby("level")["ms"].mean()

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 12.1 — contains: flag the rows whose level is ERROR *or* WARN
# (the checker runs  logs["raw"].str.contains(pattern) )
pattern = r""   # <- your pattern here
check_mask(pattern, logs["raw"], [True, False, True, True, False], how="contains")

In [ ]:
# ✏️ Exercise 12.2 — fullmatch: which strings are valid 10-digit US phone numbers?
# Accept: (555) 123-4567   555-123-4567   555.123.4567   5551234567
# (checker runs  phones.str.fullmatch(pattern) )
phones = pd.Series(["(555) 123-4567", "555-123-4567", "555.123.4567", "5551234567", "12345", "call me"])
pattern = r""   # <- your pattern here   Hint: \(? \)? and [-. ]?
check_mask(pattern, phones, [True, True, True, True, False, False], how="fullmatch")

In [ ]:
# ✏️ Exercise 12.3 — replace: strip everything that isn't a digit
# (checker runs  phones.str.replace(pattern, repl, regex=True) )
phones = pd.Series(["(555) 123-4567", "555-123-4567", "555.123.4567", "5551234567", "12345", "call me"])
pattern = r""   # <- what to remove
repl = ""
check_replace(pattern, repl, phones, ["5551234567", "5551234567", "5551234567", "5551234567", "12345", ""])

In [ ]:
# ✏️ Exercise 12.4 — extract: with NAMED groups, pull out three columns from each log line:
#   level (e.g. ERROR), ip, status (the 3-digit code).
# (checker runs  logs["raw"].str.extract(pattern) )
# Hint: after the ip come three non-space "words": method, path... then the status.
pattern = r""   # <- your pattern here
check_extract(pattern, logs["raw"], {
    "level":  ["ERROR", "INFO", "WARN", "ERROR", "INFO"],
    "ip":     ["192.168.1.15", "10.0.0.7", "192.168.1.15", "172.16.0.4", "10.0.0.7"],
    "status": ["500", "200", "404", "503", "200"],
})

In [ ]:
# ✏️ Exercise 12.5 — count: how many VALID IPv4 addresses appear in each message?
# (checker runs  messages.str.count(pattern) )
octet = r"(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)"
messages = pd.Series([
    "Failed login from 192.168.1.15 and 10.0.0.7",
    "All systems healthy",
    "Route 8.8.8.8 -> 1.1.1.1 -> 999.1.1.1",
    "src=172.16.0.4",
])
pattern = r""   # <- your pattern here (the same one as 11.3)
check_count(pattern, messages, [2, 0, 2, 1])

### 🏁 Capstone: clean a messy contact list

Each row has a name, an email and a phone number in a *different* layout. Using `.str.extract(...)` and `.str.replace(...)`, fill in the three new columns. Write each as a pandas expression on `contacts["raw"]`.

> Remember: `.str.extract(pattern)` returns a **DataFrame** with one column per capture group. With one unnamed group, take column `[0]`, like `contacts["raw"].str.extract(r"(...)")[0]`.

In [ ]:
# ✏️ Exercise 12.6 — Capstone
contacts = pd.DataFrame({"raw": [
    "Alice Smith <alice@example.com> (555) 123-4567",
    "Bob Jones bob.jones@mail.co.uk 555-987-6543",
    "carol@test.org | 555.222.3333",
    "Dave Lee 5554441111 (no email)",
]})

# 1) email: the email address (empty if the row has none)
contacts["email"] = None

# 2) phone: just the 10 digits, e.g. "5551234567"  (extract the phone, then strip non-digits)
contacts["phone"] = None

# 3) domain: the part after the @ in the email (empty if none).  Hint: @([\w.-]+)
contacts["domain"] = None

check_columns(contacts, {
    "email":  ["alice@example.com", "bob.jones@mail.co.uk", "carol@test.org", ""],
    "phone":  ["5551234567", "5559876543", "5552223333", "5554441111"],
    "domain": ["example.com", "mail.co.uk", "test.org", ""],
})
contacts

---
# 🎓 Wrap-up

## Regex cheat sheet

| Category | Syntax | Meaning |
|---|---|---|
| **Literals** | `abc` | the exact text; escape metacharacters with `\` |
| | `.` | any one character (except newline) |
| **Shorthands** | `\d` `\D` | digit / non-digit |
| | `\w` `\W` | word char (letter, digit, `_`) / non-word |
| | `\s` `\S` | whitespace / non-whitespace |
| **Classes** | `[abc]` `[a-z]` `[^abc]` | one of / range / none of |
| **Quantifiers** | `+` `*` `?` | 1+ / 0+ / 0 or 1 |
| | `{n}` `{n,m}` `{n,}` | exactly n / n to m / n or more |
| | `+?` `*?` | lazy versions |
| **Anchors** | `^` `$` `\b` | start / end / word boundary |
| **Groups** | `(...)` | capture |
| | `(?:...)` | group, don't capture |
| | `(?P<name>...)` | named capture |
| | `\1` | backreference to group 1 |
| | `a\|b` | alternation |
| **Lookaround** | `(?=...)` `(?!...)` | followed by / not followed by |
| | `(?<=...)` `(?<!...)` | preceded by / not preceded by |
| **Flags** | `re.I` `re.M` `re.S` `re.X` | ignore case / multiline `^$` / dot matches newline / verbose |

## The `re` functions

`search` (anywhere) · `match` (start) · `fullmatch` (whole string) · `findall` · `finditer` · `sub` · `split` · `compile`

## Habits that save hours

* **Raw strings** (`r"..."`) every time.
* **Validate with `fullmatch`**, extract with `findall`/`search`.
* **Build big patterns from small, tested pieces**, glued with Python strings.
* **Use `re.VERBOSE` + named groups** for anything longer than a line.
* Test **both** strings that should match **and** strings that shouldn't. That's what the checkers did for you.
* Prototype at [regex101.com](https://regex101.com) (choose the *Python* flavour). It explains your pattern piece by piece.
* Regex is for text with a *shape*. For emails/URLs/dates in production, prefer a library (`email-validator`, `urllib.parse`, `datetime`/`pd.to_datetime`) and use regex to pre-filter or extract.

## Where next?

* Redo the exercises you got wrong *without* looking at the answer key.
* Invent variants: a different date format, a different ID scheme, your own log format.
* Try `Series.str.extractall` on text with several matches per row.
* Look at `re.compile` for reusing a pattern, and `re.MULTILINE` for multi-line text.

---
# 🔑 Answer key

Try every exercise first. Then open the key to compare. Your pattern can differ from mine and still be correct, as long as the checker is happy.

<details>
<summary><b>🔑 Answer key — click to open (try first!)</b></summary>

<pre>
1.1    r&quot;at&quot;
1.2    r&quot;3\.14&quot;
1.3    r&quot;\(555\)&quot;
1.4    r&quot;c.t&quot;
2.1    r&quot;\d&quot;
2.2    r&quot;\d\d&quot;
2.3    r&quot;\d\d\d-\d\d\d\d&quot;
2.4    r&quot;\D&quot;
3.1    r&quot;\d+&quot;
3.2    r&quot;\d{4}&quot;
3.3    r&quot;\d{3}-?\d{4}&quot;
3.4    r&quot;\d{1,3}&quot;
3.5    r&quot;colou?r&quot;
4.1    r&quot;[A-Z]&quot;
4.2    r&quot;[a-z]+&quot;
4.3    r&quot;[aeiou]&quot;
4.4    r&quot;\w+&quot;
4.5    r&quot;[^,]+&quot;
5.1    r&quot;[A-Z]\d[A-Z] \d[A-Z]\d&quot;
5.2    r&quot;\(\d{3}\) \d{3}-\d{4}&quot;
5.3    r&quot;\d{4}-\d{2}-\d{2}&quot;
5.4    r&quot;[A-Za-z]\w{2,11}&quot;
5.5    r&quot;ORD-\d{8}-\d{4}&quot;
6.1    r&quot;\bcat\b&quot;
6.2    r&quot;\b\d{3}\b&quot;
6.3
       P = r&quot;\bpython\b&quot;
       F = re.IGNORECASE
6.4    r&quot;^ERROR&quot;
6.5    r&quot;\.csv$&quot;
7.1    r&quot;\w+\.(?:jpg|png|gif)&quot;
7.2    r&quot;(?:ha)+!&quot;
7.3    r&quot;(\d{1,2}):(\d{2})&quot;
7.4    r&quot;user=(?P&lt;user&gt;\w+) id=(?P&lt;id&gt;\d+)&quot;
7.5    r&quot;(?:\+1-)?\d{3}-\d{3}-\d{4}&quot;
7.6    r&quot;\b(\w+) \1\b&quot;
8.1
       P = r&quot;\d+&quot;
       R = r&quot;#&quot;
8.2
       P = r&quot;(\d{4})-(\d{2})-(\d{2})&quot;
       R = r&quot;\3/\2/\1&quot;
8.3    r&quot;\s*[,;|]\s*&quot;
8.4    r&quot;&lt;.+?&gt;&quot;
8.5
       P = r&quot;\s+&quot;
       R = r&quot; &quot;
9.1    r&quot;\d+(?= dollars)&quot;
9.2    r&quot;(?&lt;=\$)\d+(?:\.\d{2})?&quot;
9.3    r&quot;(?=.*\d)(?=.*[a-z])(?=.*[A-Z]).{8,}&quot;
9.4    r&quot;\b(?!un)\w+&quot;
10.1   r&quot;[0-9A-Fa-f]{2}&quot;
10.2   r&quot;#(?:[0-9a-fA-F]{3}){1,2}&quot;
10.3   r&quot;0[xX][0-9a-fA-F]{1,16}&quot;
10.4   r&quot;(?:[0-9A-Fa-f]{2}[:-]){5}[0-9A-Fa-f]{2}&quot;
10.5   r&quot;[0-9A-Fa-f]{2}([:-])(?:[0-9A-Fa-f]{2}\1){4}[0-9A-Fa-f]{2}&quot;
10.6   r&quot;\d{1,3}(?:\.\d{1,3}){3}&quot;
10.7   r&quot;25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d&quot;
10.8   octet + r&quot;(?:\.&quot; + octet + r&quot;){3}&quot;
10.9   ipv4 + r&quot;(?:/(?:3[0-2]|[12]?\d))?&quot;
10.10  r&quot;(?:[0-9a-fA-F]{1,4}:){7}[0-9a-fA-F]{1,4}&quot;
10.11  r&quot;[\w.+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+&quot;
10.12  r&quot;https?://[\w.-]+(?::\d{1,5})?(?:/[\w./%-]*)?(?:\?[\w=&amp;%.-]*)?&quot;
11.1   r&quot;\d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01]) (?:[01]\d|2[0-3]):[0-5]\d:[0-5]\d&quot;
11.2   r&quot;&quot;&quot;
    (?P&lt;date&gt;\d{4}-\d{2}-\d{2})      \s+
    (?P&lt;time&gt;\d{2}:\d{2}:\d{2})      \s+
    \[ (?P&lt;level&gt;[A-Z]+) \]          \s+
    (?P&lt;ip&gt;\d{1,3}(?:\.\d{1,3}){3})  \s+
    (?P&lt;method&gt;GET|POST|PUT|DELETE)  \s+
    (?P&lt;path&gt;\S+)                    \s+
    (?P&lt;status&gt;\d{3})                \s+
    (?P&lt;ms&gt;\d+) ms
&quot;&quot;&quot;
11.3   r&quot;\b&quot; + octet + r&quot;(?:\.&quot; + octet + r&quot;){3}\b&quot;
11.4
       DEV = r&quot;DEV-[0-9A-F]{6}&quot;
       TS = r&quot;\d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])T(?:[01]\d|2[0-3]):[0-5]\d:[0-5]\dZ&quot;
       IP = octet + r&quot;(?:\.&quot; + octet + r&quot;){3}&quot;
       MAC = r&quot;(?:[0-9A-Fa-f]{2}:){5}[0-9A-Fa-f]{2}&quot;
       ADDR = r&quot;0x[0-9A-Fa-f]{8}&quot;
       READ = r&quot;[a-z]+=-?\d+(?:\.\d+)?[A-Za-z%]*&quot;
11.5   r&quot;(?P&lt;ORDER&gt;ORD-\d{8}-\d{4})|(?P&lt;IP&gt;\d{1,3}(?:\.\d{1,3}){3})|(?P&lt;DATE&gt;\d{4}-\d{2}-\d{2})|(?P&lt;MONEY&gt;\$\d{1,3}(?:,\d{3})*(?:\.\d{2})?)|(?P&lt;HEX&gt;0[xX][0-9A-Fa-f]+)&quot;
12.1   r&quot;\[(?:ERROR|WARN)\]&quot;
12.2   r&quot;\(?\d{3}\)?[-. ]?\d{3}[-. ]?\d{4}&quot;
12.3   r&quot;\D&quot;
12.4   r&quot;\[(?P&lt;level&gt;[A-Z]+)\]\s+(?P&lt;ip&gt;\S+)\s+\S+\s+\S+\s+(?P&lt;status&gt;\d{3})&quot;
12.5   r&quot;\b&quot; + octet + r&quot;(?:\.&quot; + octet + r&quot;){3}\b&quot;
12.6
       EMAIL = contacts[&quot;raw&quot;].str.extract(r&quot;([\w.+-]+@[\w-]+(?:\.[\w-]+)+)&quot;)[0]
       PHONE = contacts[&quot;raw&quot;].str.extract(r&quot;(\(?\d{3}\)?[-. ]?\d{3}[-. ]?\d{4})&quot;)[0].str.replace(r&quot;\D&quot;, &quot;&quot;, regex=True)
       DOMAIN = contacts[&quot;raw&quot;].str.extract(r&quot;@([\w.-]+)&quot;)[0]
</pre>

</details>